# Fase 3 — Pré-processamentoEntrega a matriz que vai para a clusterização e a **Figura 3** (variânciaexplicada acumulada). Como a clusterização ficou para a entrega seguinte,esta fase é o **último passo técnico** do relatório parcial — e por isso o queaqui era preparação passa a ser resultado.O plano previa quatro passos: `log1p`, `RobustScaler` ajustado só nosubconjunto clusterizado, peso `1/√n` por bloco e PCA. Os três primeirosinteragem de um jeito que só aparece quando se mede, e é isso que estenotebook mostra.

In [ ]:
import sysfrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.preprocessing import RobustScaler, StandardScalerfrom sklearn.decomposition import PCARAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(RAIZ / "src"))from estilo import (aplicar_estilo, salvar, AZUL, LARANJA, MUDO, TINTA_2,                    GRADE, CMAP_SEQUENCIAL)from config import BASE_FINAL_CSV, DICIONARIO_CSV, DATA_PROCESSEDaplicar_estilo()print(f"Python: {sys.executable}")base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})dic = pd.read_csv(DICIONARIO_CSV)bloco_de = dic.set_index("coluna")["bloco"]ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))# Passo 2 do plano: fora quem não tem IEGM (a capital).mod = base[~base["flag_sem_iegm"]].copy().reset_index(drop=True)N_BLOCO = {b: sum(1 for c in features if bloco_de[c] == b)           for b in ORDEM_BLOCOS}print(f"{len(mod)} municípios x {len(features)} features  {N_BLOCO}")

## 1. `log1p` nas taxas e no PIB per capitaJustificado na Fase 2: as distribuições brutas têm assimetria de +1,1 a +5,0,e sem transformação a distância euclidiana seria dominada pelos poucosmunicípios de taxa extrema — que, como aquela fase mostrou, são os depopulação minúscula, não os mais violentos.`taxa_urbanizacao` fica **fora** do `log1p`: é um percentual limitado a 100,já simétrico, e a transformação só o distorceria. As 8 ordinais do IEGMtambém ficam de fora, pela mesma razão.

In [ ]:
log_cols = [c for c in features            if c.startswith("taxa_") and c != "taxa_urbanizacao"] \           + ["pib_percapita"]X = mod[features].copy()X[log_cols] = np.log1p(X[log_cols])print(f"log1p em {len(log_cols)} colunas: "      f"{', '.join(c.replace('taxa_', '') for c in log_cols)}")print(f"fora do log1p: "      f"{', '.join(c for c in features if c not in log_cols)}")

## 2. O peso por bloco só funciona com o escalonador certoO peso `1/√n` existe para que os três blocos disputem a distância em pé deigualdade: sem ele, gestão (8 colunas) e criminalidade (9) esmagariamsocioeconômico (2) por acaso do número de colunas.Mas a dedução do `1/√n` supõe uma coisa que costuma passar despercebida:**que cada coluna padronizada contribua com uma unidade de variância.** Seuma coluna sai da padronização com variância 3, ela leva três vezes o pesoque o cálculo lhe atribuiu. Vale medir em vez de supor.Comparamos três escalonamentos, todos com o mesmo `log1p` e o mesmo `1/√n`:| | Contínuas | Ordinais do IEGM ||---|---|---|| **A** | `RobustScaler` | `RobustScaler` (o plano original) || **B** | `RobustScaler` | escala teórica 1–5, sem reescalar || **C** | `StandardScaler` | `StandardScaler` |

In [ ]:
ordinais = [c for c in features if bloco_de[c] == "gestao"]continuas = [c for c in features if c not in ordinais]pesos = {b: 1 / np.sqrt(N_BLOCO[b]) for b in ORDEM_BLOCOS}vetor_peso = [pesos[bloco_de[c]] for c in features]print("pesos 1/sqrt(n):", {b: round(v, 3) for b, v in pesos.items()})def ponderar(Z):    return Z.mul(vetor_peso, axis=1)def orcamento(W):    """Quanto da variância total cada bloco leva, em %."""    var = W.var(ddof=0)    por_bloco = var.groupby(W.columns.map(bloco_de)).sum().reindex(ORDEM_BLOCOS)    return (por_bloco / por_bloco.sum() * 100).round(1)def escalar(nome):    Z = pd.DataFrame(index=X.index, columns=features, dtype=float)    if nome == "A":        Z[features] = RobustScaler().fit_transform(X[features])    elif nome == "B":        Z[continuas] = RobustScaler().fit_transform(X[continuas])        # Escala teórica: 1..5 vira 0..1, apenas centrada.        Z[ordinais] = (X[ordinais] - 1) / 4.0        Z[ordinais] = Z[ordinais] - Z[ordinais].mean()    else:        Z[features] = StandardScaler().fit_transform(X[features])    return Zcomparacao = {}for nome in ["A", "B", "C"]:    W = ponderar(escalar(nome))    ev = PCA().fit(W).explained_variance_ratio_    carga_pc2 = pd.Series(PCA().fit(W).components_[1],                          index=features).abs().sort_values(ascending=False)    comparacao[nome] = dict(        orcamento(W),        PC1=round(ev[0] * 100, 1),        PC2=round(ev[1] * 100, 1),        comps_80=int(np.argmax(np.cumsum(ev) >= 0.80)) + 1,        dominante_PC2=f"{carga_pc2.index[0]} ({carga_pc2.iloc[0]:.2f})",    )pd.DataFrame(comparacao).T

### O que a tabela mostra**A (RobustScaler em tudo) não equaliza e ainda cria um eixo fantasma.** Oorçamento fica em 37 / 33 / 30 em vez de 33 / 33 / 33, e a segunda componenteprincipal — quase 15% da variância — é **uma única variável**,`i_planejamento_ord`, com carga 0,88. Um bloco inteiro de 8 dimensõesresumido a um eixo que na prática é uma coluna só.A causa está na aritmética do `RobustScaler`: ele divide pelo intervalointerquartil, e o IQR de `i_planejamento_ord` é **0,333** — 70% dosmunicípios estão no mesmo valor. Dividir por 0,333 multiplica a coluna portrês. O escalonador amplifica justamente a variável com menos informação.**B (escala teórica nas ordinais) erra para o outro lado.** Preservar aescala 1–5 derruba o bloco de gestão para 2,3% do orçamento. Como integrargestão pública é a contribuição que o artigo reivindica, esvaziá-la a 2% nãoé opção.**C (StandardScaler) entrega exatamente o que o `1/√n` promete:** 33,3% paracada bloco. Não é coincidência — `StandardScaler` deixa toda coluna comvariância 1, que é precisamente a suposição sobre a qual o peso `1/√n` foideduzido. E nenhuma variável sequestra uma componente: a maior carga na PC2cai de 0,88 para 0,38.**Decisão: `log1p` + `StandardScaler` + peso `1/√n`.** É um desvio do plano,que previa `RobustScaler`. O motivo do plano era conter os valores extremos —mas isso quem faz é o `log1p`, e a Fase 2 mostrou que ele já deixa aassimetria entre −0,5 e +0,7. Aplicar `RobustScaler` em cima disso nãoprotege de nada e quebra a equalização dos blocos.

In [ ]:
Z = escalar("C")W = ponderar(Z)print(f"matriz final: {W.shape[0]} x {W.shape[1]}")print(f"nulos: {int(W.isna().sum().sum())} | "      f"infinitos: {int(np.isinf(W.to_numpy()).sum())}")orcamento(W)

### O efeito do peso, em uma figuraSem peso, o número de colunas que cada fonte por acaso publica decide quantoela influencia a distância.

In [ ]:
sem_peso = orcamento(Z)com_peso = orcamento(W)fig, ax = plt.subplots(figsize=(7.0, 3.4))x = np.arange(len(ORDEM_BLOCOS))larg = 0.34b1 = ax.bar(x - 0.18, [sem_peso[b] for b in ORDEM_BLOCOS], larg,            label="sem peso", color=AZUL)b2 = ax.bar(x + 0.18, [com_peso[b] for b in ORDEM_BLOCOS], larg,            label="com peso 1/√n", color=LARANJA)ax.bar_label(b1, fmt="%.1f%%", fontsize=8, color=TINTA_2, padding=2)ax.bar_label(b2, fmt="%.1f%%", fontsize=8, color=TINTA_2, padding=2)# Margem à direita reservada só para o rótulo da linha de paridade: assim ele# não disputa espaço com o valor da última barra.ax.set_xlim(-0.6, 3.05)ax.axhline(100 / 3, color=MUDO, lw=1, ls=(0, (4, 3)), zorder=0)ax.text(2.52, 100 / 3 + 1.2, "33,3%\nparidade", fontsize=7.5, color=MUDO,        ha="left", va="bottom")ax.set_xticks(x, [f"{b}\n(n={N_BLOCO[b]})" for b in ORDEM_BLOCOS])ax.set_ylabel("% do orçamento de distância")ax.set_ylim(0, 56)ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.02), ncols=2)ax.set_title("Participação de cada bloco na distância, antes e depois do peso",             pad=22)fig.tight_layout()salvar(fig, "figura_orcamento_blocos")plt.show()

## 3. Figura 3 — PCA e variância explicada acumulada

In [ ]:
pca = PCA().fit(W)ev = pca.explained_variance_ratio_cum = np.cumsum(ev)k80 = int(np.argmax(cum >= 0.80)) + 1fig, ax = plt.subplots(figsize=(7.0, 3.6))comps = np.arange(1, len(ev) + 1)ax.bar(comps, ev * 100, color=GRADE, label="variância da componente")ax.plot(comps, cum * 100, color=AZUL, marker="o", markersize=4,        label="acumulada")ax.axhline(80, color=MUDO, lw=1, ls=(0, (4, 3)), zorder=0)ax.annotate(f"80% da variância\ncom {k80} componentes",            xy=(k80, 80), xytext=(k80 + 0.6, 55), fontsize=8, color=TINTA_2,            arrowprops=dict(arrowstyle="-", color=MUDO, lw=0.9))ax.set_xticks(comps)ax.set_xlabel("componente principal")ax.set_ylabel("% da variância")ax.set_ylim(0, 102)ax.legend(loc="center right")ax.set_title("Variância explicada pelo PCA sobre a matriz ponderada")fig.tight_layout()salvar(fig, "figura3_variancia_pca")plt.show()pd.DataFrame({"componente": comps, "variância %": (ev * 100).round(1),              "acumulada %": (cum * 100).round(1)}).head(12)

**Este é o resultado mais importante da fase, e não é o esperado.** Sãonecessárias **9 componentes para 80% da variância** — de 19 possíveis. Não háestrutura de baixa dimensão: a informação está genuinamente espalhada.Duas leituras, e as duas importam:1. **A favor do desenho.** Se os três blocos fossem redundantes, poucas   componentes bastariam. Não bastam — o que confirma, agora por outro   caminho, o mesmo que a matriz de Spearman da Fase 2 já indicava: crime,   condição socioeconômica e gestão medem coisas diferentes. Integrar as três   fontes acrescenta informação em vez de repeti-la.2. **Um aviso para a próxima entrega.** Alta dimensionalidade intrínseca é   terreno ruim para clusterização baseada em densidade. É bem provável que o   DBSCAN rotule quase tudo como ruído — e isso passa a ser uma **previsão   feita antes de rodar**, com um número por trás, e não uma desculpa   inventada depois.

## 4. O que as primeiras componentes significam

In [ ]:
cargas = pd.DataFrame(pca.components_[:3].T, index=features,                      columns=["PC1", "PC2", "PC3"]).round(3)cargas["bloco"] = [bloco_de[c] for c in features]for pc in ["PC1", "PC2", "PC3"]:    forte = cargas.loc[cargas[pc].abs() > 0.25, [pc, "bloco"]]    print(f"\n{pc} ({ev[int(pc[-1]) - 1] * 100:.1f}% da variância)")    for c, linha in forte.reindex(            forte[pc].abs().sort_values(ascending=False).index).iterrows():        print(f"  {linha[pc]:+.3f}  {c:26s} [{linha['bloco']}]")

In [ ]:
escores = pca.transform(W)fig, ax = plt.subplots(figsize=(6.4, 5.0))p = ax.scatter(escores[:, 0], escores[:, 1], c=mod["taxa_urbanizacao"],               cmap=CMAP_SEQUENCIAL, s=16, linewidths=0)ax.grid(False)ax.axhline(0, color=GRADE, lw=0.8, zorder=0)ax.axvline(0, color=GRADE, lw=0.8, zorder=0)ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")cb = fig.colorbar(p, ax=ax, shrink=0.75)cb.set_label("taxa de urbanização (%)", fontsize=8)cb.outline.set_visible(False)ax.set_title("Os 644 municípios nas duas primeiras componentes")fig.tight_layout()salvar(fig, "figura_pc1_pc2")plt.show()

A nuvem é **contínua**: um gradiente, não ilhas separadas. A cor mostra que aPC1 é em boa medida um eixo de urbanização e porte econômico, e os municípiosse distribuem ao longo dele sem descontinuidade visível.Isso não invalida a clusterização — k-means particiona nuvens contínuas e oresultado pode ser útil como tipologia. Mas antecipa que os grupos serão**cortes num gradiente**, e não agrupamentos naturalmente separados, e que asilhueta tende a ser modesta. Melhor saber disso agora do que interpretarmal um coeficiente baixo na próxima entrega.

## 5. Matriz exportada`matriz_modelagem.csv` guarda a matriz pronta — já filtrada nos 644, com`log1p`, padronizada e ponderada. É o insumo direto da clusterização, egravá-la garante que a próxima entrega parta exatamente desta, sem repetirdecisões de memória.

In [ ]:
saida = W.copy()saida.insert(0, "codigo_ibge", mod["codigo_ibge"].values)saida.insert(1, "municipio", mod["municipio"].values)destino = DATA_PROCESSED / "matriz_modelagem.csv"saida.to_csv(destino, index=False, encoding="utf-8")print(f"-> {destino.name}: {saida.shape[0]} linhas x {saida.shape[1]} colunas")pd.DataFrame({    "passo": ["subconjunto", "log1p", "escalonamento", "peso por bloco",              "PCA"],    "decisão": [        f"{len(mod)} municípios (exclui flag_sem_iegm)",        f"{len(log_cols)} colunas (taxas + PIB per capita)",        "StandardScaler, ajustado só neste subconjunto",        "1/√n: " + ", ".join(f"{b}={pesos[b]:.3f}" for b in ORDEM_BLOCOS),        f"{k80} componentes para 80% da variância",    ],})

## 6. O que fica decidido1. **`log1p` + `StandardScaler` + peso `1/√n`.** Desvio deliberado do plano,   que previa `RobustScaler`: só com `StandardScaler` o peso `1/√n` entrega a   paridade de 33,3% que ele promete, e só assim nenhuma variável isolada   captura uma componente principal.2. **`RobustScaler` produzia um artefato**, não um resultado: `i_planejamento_ord`   respondia sozinha por 88% da segunda componente porque seu IQR de 0,333 —   70% dos municípios no mesmo valor — fazia o escalonador multiplicá-la por   três.3. **Nenhuma feature foi descartada.** As 19 seguem, coerente com a Fase 2,   que não encontrou par com |ρ| > 0,85.4. **São necessárias 9 componentes para 80% da variância.** Evidência a favor   da integração das três fontes e alerta antecipado sobre a dificuldade da   clusterização por densidade.5. **`i_planejamento_ord` fica sob observação.** Com 70% dos municípios no   mesmo valor, ela carrega pouca informação em qualquer escalonamento. Vale   uma análise de sensibilidade com e sem ela na próxima entrega — não uma   exclusão silenciosa agora.